<a href="https://colab.research.google.com/github/TinevimboMusingadi/J-nuerons/blob/main/notebooks/jneurons_qwen_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# J-Neurons: Real-Time Jailbreak Detection via Sparse Compliance Neurons in Multilingual African LLM Deployments

This notebook provides a complete end-to-end evaluation pipeline for identifying and monitoring **J-Neurons** (Jailbreak/Compliance Neurons) in open-weight models (Qwen-2.5-1.5B-Instruct).
We evaluate cross-lingual transfer of J-neuron activation profiles to low-resource African languages (Afrikaans, Kiswahili, isiZulu, and Shona) and implement a real-time token-level **circuit breaker** to halt jailbreaks before they complete.

**Author:** King Tine (HIT)  
**Hackathon track:** Global South AI Safety Hackathon (Africa Track)


## Setup & Installation
First, we install all necessary libraries and dependencies.


In [ ]:
# Detect if running in Google Colab
import sys
if 'google.colab' in sys.modules:
    print('Running in Google Colab. Cloning repository...')
    !git clone https://github.com/TinevimboMusingadi/J-nuerons.git
    %cd J-nuerons
else:
    print('Running locally. No repository clone needed.')

# Install core ML and translation packages
!pip install -q transformers accelerate scikit-learn matplotlib pandas numpy google-generativeai tqdm

# Create directories to keep outputs neat
import os
os.makedirs('results', exist_ok=True)
os.makedirs('reports', exist_ok=True)


## Step 1: Load Qwen-2.5-1.5B-Instruct
We load the target open-weight model and its tokenizer in half-precision (float16) to ensure low memory consumption.


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'Loading model: {model_id}...')
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True
)
print('Model loaded successfully!')


Loading model: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!


## Step 2: Prompt Dataset Generation
We compile our dataset of harmful prompts, benign controls, and target African language translations. We can use the Gemini API (if `GEMINI_API_KEY` is present) or fallback to our pre-translated dataset in `src/data/generate_data.py`.


In [13]:
import os
import subprocess
from google.colab import userdata

# 1. Explicitly load the API key from Colab Secrets into the environment
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print("API Key found and loaded into environment.")
except userdata.SecretNotFoundError:
    print("Warning: GEMINI_API_KEY not found in Colab Secrets. Make sure the toggle is turned on!")

# 2. Run the dataset generation script
print('Generating prompt dataset...')
result = subprocess.run(['python', '-m', 'src.data.generate_data'], capture_output=True, text=True)

# 3. Print results
print("\n--- Output ---")
print(result.stdout)

if result.stderr:
    print('--- Warnings/Errors ---')
    print(result.stderr)

API Key found and loaded into environment.
Generating prompt dataset...

--- Output ---
GEMINI_API_KEY detected. Generating synthetic prompts via Gemini API...
Error calling Gemini for cyberattacks: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]
Error calling Gemini for malware: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]
Error calling Gemini for harassment: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key

## Step 3: Extract CETT Activations
We load the dataset, register CETT activation tracking hooks on all layers, and collect features across our prompts.


In [14]:
import json
import importlib
from src.hooks import CETTHookManager
import src.probe

# 1. Force reload the patched module
importlib.reload(src.probe)
from src.probe import extract_cett_features

# Load dataset
with open('src/data/datasets/jneurons_dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

hook_manager = CETTHookManager(model)

# Prepare training data
train_prompts = []
for item in dataset['harmful']:
    train_prompts.append({'prompt': item['en'], 'is_harmful': True, 'is_jailbreak': False})
    for template_info in dataset['jailbreaks']:
        jailbroken_prompt = template_info['template'].format(prompt=item['en'])
        train_prompts.append({'prompt': jailbroken_prompt, 'is_harmful': True, 'is_jailbreak': True})

for item in dataset['benign']:
    train_prompts.append({'prompt': item['en'], 'is_harmful': False, 'is_jailbreak': False})

print(f'Extracting CETT features for {len(train_prompts)} English prompts...')

# 2. Run the patched function
X_train, y_train, train_texts = extract_cett_features(model, tokenizer, hook_manager, train_prompts)

print('Feature extraction completed!')
print(f'Features shape: {X_train.shape}')
print(f'Compliance labels (Class 1 counts): {sum(y_train)} / {len(y_train)}')

for i in range(min(5, len(train_prompts))):
    print(f'Prompt: {train_prompts[i]["prompt"][:60]}...')
    print(f'Label: {y_train[i]} | Response: {train_texts[i][:80]}...\n')

Extracting CETT features for 25 English prompts...


Extracting CETT features:   0%|          | 0/25 [00:00<?, ?it/s]


AttributeError: 

## Step 4: Identify J-Neurons (L1-Regularized Logistic Regression)
We fit our sparse logistic regression classifier. L1 regularization acts as a feature selector, isolating a tiny subset of J-Neurons.


In [ ]:
from src.probe import train_probe, evaluate_probe

print('Training sparse probe...')
clf, j_neurons = train_probe(X_train, y_train, C=0.5)

print(f'Total J-Neurons found with non-zero weights: {len(j_neurons)}')
print('\nTop 15 J-Neurons by weight magnitude:')
for i, neuron in enumerate(j_neurons[:15]):
    print(f"{i+1}. Layer {neuron['layer_idx']}, Neuron {neuron['neuron_idx']} (Weight: {neuron['weight']:.4f})")

train_auroc, _ = evaluate_probe(clf, X_train, y_train)
print(f'\nIn-Distribution (English) AUROC: {train_auroc:.4f}')


## Step 5: Cross-Lingual Transfer Evaluation
We evaluate the English-trained J-Neuron probe against translations in Afrikaans (af), Kiswahili (sw), isiZulu (zu), and Shona (sn) to test cross-lingual universality.


In [ ]:
from src.eval.plots import plot_cross_lingual_roc

languages = ['en', 'af', 'sw', 'zu', 'sn']
lang_labels = {}
lang_probs = {}

for lang in languages:
    print(f'Evaluating J-Neuron probe cross-lingually on language: {lang}...')
    lang_prompts = []

    # Build evaluation set for this language
    for item in dataset['harmful']:
        lang_prompts.append({
            'prompt': item[lang],
            'is_harmful': True,
            'is_jailbreak': False
        })
        for template_info in dataset['jailbreaks']:
            # Multilingual jailbreak template wrapping
            jailbroken_prompt = template_info['template'].format(prompt=item[lang])
            lang_prompts.append({
                'prompt': jailbroken_prompt,
                'is_harmful': True,
                'is_jailbreak': True
            })
    for item in dataset['benign']:
        lang_prompts.append({
            'prompt': item[lang],
            'is_harmful': False,
            'is_jailbreak': False
        })

    X_lang, y_lang, _ = extract_cett_features(model, tokenizer, hook_manager, lang_prompts)
    auroc, probs = evaluate_probe(clf, X_lang, y_lang)
    print(f'[{lang.upper()}] Evaluation AUROC: {auroc:.4f}')

    lang_labels[lang] = y_lang
    lang_probs[lang] = probs

# Plot ROC Curves and display inline
fig_roc = plot_cross_lingual_roc(lang_labels, lang_probs)
fig_roc.show()


## Step 6: Mechanical Interpretability — J-Neuron Logit Lens
We project the output weights ($W_{down}$) of the top J-neurons through the model's final norm and LM head. This allows us to inspect what tokens these neurons directly boost in the model's vocabulary.


In [ ]:
from src.interp.logit_lens import get_neuron_vocabulary_projection

print('Running Logit Lens on top J-neurons:')
for i, neuron in enumerate(j_neurons[:5]):
    layer = neuron['layer_idx']
    n_idx = neuron['neuron_idx']
    top_tokens = get_neuron_vocabulary_projection(model, tokenizer, layer, n_idx, k=8)
    token_strs = [f'"{t}" ({l:.2f})' for t, l in top_tokens]
    print(f'\nJ-Neuron #{i+1} (L{layer}, N{n_idx}):')
    print('  Boosted tokens:', ', '.join(token_strs))


## Step 7: Real-Time Circuit Breaker Performance
We configure the token-level circuit breaker using the discovered J-Neurons. We compare the model response and jailbreak success rate with vs. without circuit-breaker stopping.


In [ ]:
from src.circuit_breaker import JNeuronCircuitBreaker
from src.eval.plots import plot_asr_comparison
from src.probe import is_refusal

# We configure the circuit breaker with the top 15 J-neurons and standard logistic regression bias
bias = float(clf.intercept_[0])
circuit_breaker = JNeuronCircuitBreaker(
    model=model,
    tokenizer=tokenizer,
    j_neurons=j_neurons[:15],
    intercept_threshold=0.5,
    intercept_bias=bias
)

# Evaluate Jailbreak Success Rate (ASR) before vs after circuit breaker
asr_before = {}
asr_after = {}

for lang in languages:
    # Get jailbroken test prompts
    test_prompts = []
    for item in dataset['harmful']:
        for template_info in dataset['jailbreaks']:
            test_prompts.append(template_info['template'].format(prompt=item[lang]))

    success_before = 0
    success_after = 0

    for prompt in test_prompts[:5]: # Sub-sample for quick execution
        # 1. Before: run normal generation
        # Standard chat prompt formatting
        inputs = tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(inputs, max_new_tokens=30, pad_token_id=tokenizer.eos_token_id)
        text_before = tokenizer.decode(out[0, inputs.shape[1]:], skip_special_tokens=True)
        if not is_refusal(text_before):
            success_before += 1

        # 2. After: run circuit breaker generation
        res = circuit_breaker.generate(prompt, max_new_tokens=30, return_scores=True)
        text_after = res['completion']
        # If the generation was halted, we consider the jailbreak successfully blocked (not compliant)
        if not res['interrupted'] and not is_refusal(text_after):
            success_after += 1

    asr_before[lang] = success_before / 5
    asr_after[lang] = success_after / 5
    print(f'[{lang.upper()}] ASR Before: {asr_before[lang]*100:.1f}% | ASR After: {asr_after[lang]*100:.1f}%')

# Plot and display comparison bar chart
fig_asr = plot_asr_comparison(asr_before, asr_after)
fig_asr.show()


## Step 8: (Stretch) Causal Scaling Sweep
We scale the activations of the J-neurons during generation by scaling factor $\alpha \in [0, 2.5]$ to demonstrate their causal control on jailbreak compliance rates.


In [ ]:
from src.eval.plots import plot_causal_scaling
from src.interp.patching import patch_activations_and_generate

# Sweep alpha scaling factors
alphas = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5]
compliance_rates = []

test_prompt = dataset['harmful'][0]['en']
jailbroken_test = dataset['jailbreaks'][0]['template'].format(prompt=test_prompt)

for alpha in alphas:
    # Create J-neuron donor activations based on scaling
    donor_acts = {}
    for n in j_neurons[:15]:
        # Scale neuron outputs dynamically during activation hook pass
        # Normal activation is scaled by alpha
        # (For this mock patch check, we override values with constant or scaled profiles)
        donor_acts[(n['layer_idx'], n['neuron_idx'])] = torch.tensor(alpha * 1.5).to(model.device)

    comp_text = patch_activations_and_generate(
        model=model,
        tokenizer=tokenizer,
        prompt=jailbroken_test,
        j_neurons=j_neurons[:15],
        donor_activations=donor_acts,
        max_new_tokens=25
    )

    compliant = not is_refusal(comp_text)
    compliance_rates.append(1.0 if compliant else 0.0)
    print(f'Alpha: {alpha:.1f} | Compliant completion: "{comp_text[:60]}..."')


## Pipeline Completed Successfully!
All figures and reports are exported inside `results/` and `reports/` directories.
